# V5: Selective Z-Mixing Grid Search

V4'un en iyi config'i uzerine **selective mixing A/B testi**.

- **Uniform alpha**: tum (i,j) pair'lerine ayni alpha uygulanir
- **Per-pair adaptive alpha**: hareket eden pair'ler yuksek alpha, sabit pair'ler dusuk/sifir alpha

**Fazlar:**
- Phase A: Uniform vs Selective baseline (2 run)
- Phase B: `change_cutoff` sweep (5 run)
- Phase C: `alpha_base` x `alpha_max` sweep (6 run)
- Phase D: Mapping & weight sweep (9 run)
- Analiz: TM-score, RMSD, alpha_mask heatmap, change_score diagnostics

**Toplam:** ~22 run

## 1. Environment Setup (bir kere calistir)

In [ ]:
import os, sys, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch pandas

sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print('Environment setup complete.')

## 2. Config

In [ ]:
# ════════════════════════════════════════════════════
#  CONFIG - V5
#  V4'ten sabit parametreler + V5 selective mixing sweep
# ════════════════════════════════════════════════════

# -- PDB --
INITIAL_PDB = '1AKE'
TARGET_PDB  = '4AKE'
CHAIN_ID    = 'A'

# -- Pipeline (V4'ten sabit) --
N_STEPS              = 20
COMBINATION_STRATEGY = 'autostop'
Z_MIXING_ALPHA       = 0.7
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# -- OF3 --
USE_MSA_SERVER        = True
USE_TEMPLATES         = False
NUM_ROLLOUT_STEPS     = None
NUM_DIFFUSION_SAMPLES = 1

# -- Confidence (V3/V4'ten sabit) --
ALPHA_DECAY               = 0.8
MAX_STALL                 = 8
CONFIDENCE_PTM_CUTOFF     = 0.50
CONFIDENCE_PLDDT_CUTOFF   = 40.0
CONFIDENCE_RANKING_CUTOFF = 0.10
CONFIDENCE_RG_MAX         = 2.0
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True
CONFIDENCE_RMSD_INIT_MAX  = 10.0

# -- Drift korumalari (V4'ten sabit) --
ENABLE_BEST_ROLLBACK  = True
BEST_ROLLBACK_TM_DROP = 0.40
ENABLE_ADAPTIVE_STOP  = True
ADAPTIVE_STOP_WINDOW  = 3

# -- Fallback --
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# -- Autostop --
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# -- Converter --
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

# ════════════════════════════════════════════════════
#  V5 YENI: Selective Mixing Sweep Parametreleri
# ════════════════════════════════════════════════════

SELECTIVE_CHANGE_CUTOFFS = [0.05, 0.1, 0.15, 0.2, 0.3]
SELECTIVE_ALPHA_BASES    = [0.0, 0.05]
SELECTIVE_ALPHA_MAXES    = [0.5, 0.7, 1.0]
SELECTIVE_MAPPINGS       = ['linear', 'sigmoid', 'step']
SELECTIVE_W_PAIRS        = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]  # (w_c, w_d)

print('V5 Config ready.')

## 3. Converter + PDB + OF3 yukleme

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import torch
import time
import copy
import pandas as pd

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.composite_confidence import (
    CompositeWeights, compute_composite, composite_score_from_step,
    WEIGHT_PRESETS, THRESHOLD_PRESETS,
    normalize_ptm, normalize_plddt, normalize_pae, normalize_rg, normalize_contact_recon,
)
from src.selective_mixing import compute_change_score, compute_alpha_mask, selective_blend_z

# -- Checkpoint --
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded from {CHECKPOINT_PATH}')

In [ ]:
# -- PDB --
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

def fetch_ca(pdb_id, chain_id):
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence

initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
target_ca, _ = fetch_ca(TARGET_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'Initial: {INITIAL_PDB} chain {CHAIN_ID}, N={N}')
print(f'Target:  {TARGET_PDB} chain {CHAIN_ID}, N={target_ca.shape[0]}')
print(f'RMSD to target: {compute_rmsd(initial_ca, target_ca):.2f} A')
print(f'TM-score: {tm_score(initial_ca, target_ca):.4f}')

# -- OF3 --
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)
_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{
                'molecule_type': 'protein',
                'chain_ids': [CHAIN_ID],
                'sequence': sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion
diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')

from src.autostop_adapter import StructureContext
_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print(f'StructureContext: N={structure_ctx.struct.N}')
print('OF3 ready.')

## 4. V5 Runner (V4 safeguard'lari + selective mixing destegi)

In [ ]:
# ════════════════════════════════════════════════════
#  V5 RUNNER
#  V4 runner + selective mixing parametreleri.
#  selective_mixing=False ise V4 ile aynisini yapar.
# ════════════════════════════════════════════════════

def make_v5_config(
    n_steps=N_STEPS,
    alpha=Z_MIXING_ALPHA,
    alpha_decay=ALPHA_DECAY,
    max_stall=MAX_STALL,
    ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
    plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
    ranking_cutoff=CONFIDENCE_RANKING_CUTOFF,
    selective_mixing=False,
    selective_params=None,
):
    """V5 config: V4 + selective mixing."""
    sel = selective_params or {}
    return ModeDriveConfig(
        n_steps=n_steps,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=alpha,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        confidence_ptm_cutoff=ptm_cutoff,
        confidence_plddt_cutoff=plddt_cutoff,
        confidence_ranking_cutoff=ranking_cutoff,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=CHAIN_ID,
        rejected_alpha_decay=alpha_decay,
        max_consecutive_rejected=max_stall,
        confidence_warmup_steps=0,
        # V4 drift korumalari
        enable_best_rollback=ENABLE_BEST_ROLLBACK,
        best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
        enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
        adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
        # V5 selective mixing
        selective_mixing=selective_mixing,
        selective_w_connectivity=sel.get('w_connectivity', 0.5),
        selective_w_distance=sel.get('w_distance', 0.5),
        selective_change_cutoff=sel.get('change_cutoff', 0.1),
        selective_alpha_base=sel.get('alpha_base', 0.0),
        selective_alpha_max=sel.get('alpha_max', 1.0),
        selective_mapping=sel.get('mapping', 'linear'),
        selective_distance_mode=sel.get('distance_mode', 'max'),
    )


def run_v5(
    weights: CompositeWeights,
    threshold: float,
    label: str = '',
    n_steps: int = N_STEPS,
    alpha: float = Z_MIXING_ALPHA,
    alpha_decay: float = ALPHA_DECAY,
    max_stall: int = MAX_STALL,
    early_abort_steps: int = 5,
    selective_mixing: bool = False,
    selective_params: dict | None = None,
    verbose: bool = True,
) -> dict:
    """V5 runner: V4 composite score + drift korumalari + selective mixing.

    selective_params dict ornegi:
        {
            'change_cutoff': 0.1,
            'alpha_base': 0.0,
            'alpha_max': 1.0,
            'mapping': 'linear',
            'w_connectivity': 0.5,
            'w_distance': 0.5,
            'distance_mode': 'max',
        }
    """
    cfg = make_v5_config(
        n_steps=n_steps, alpha=alpha,
        alpha_decay=alpha_decay, max_stall=max_stall,
        selective_mixing=selective_mixing,
        selective_params=selective_params,
    )
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    sel_tag = 'SELECTIVE' if selective_mixing else 'UNIFORM'
    sel_info = ''
    if selective_mixing and selective_params:
        sp = selective_params
        sel_info = (
            f'  selective: cutoff={sp.get("change_cutoff", 0.1)} '
            f'base={sp.get("alpha_base", 0.0)} '
            f'max={sp.get("alpha_max", 1.0)} '
            f'map={sp.get("mapping", "linear")} '
            f'w=({sp.get("w_connectivity", 0.5)},{sp.get("w_distance", 0.5)})'
        )

    print(f'\n{"="*70}')
    print(f'  {label} [{sel_tag}]')
    print(f'  weights: {weights.as_dict()}')
    print(f'  threshold={threshold:.2f}  alpha={alpha}  decay={alpha_decay}  stall={max_stall}')
    if sel_info:
        print(sel_info)
    print(f'  safeguards: rmsd_init_max={CONFIDENCE_RMSD_INIT_MAX}  '
          f'rollback={ENABLE_BEST_ROLLBACK}(drop={BEST_ROLLBACK_TM_DROP})  '
          f'adaptive_stop={ENABLE_ADAPTIVE_STOP}(w={ADAPTIVE_STOP_WINDOW})')
    print(f'{"="*70}')

    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = alpha
    current_alpha = alpha
    consecutive_rejected = 0

    # Best-so-far tracking
    coords_best = initial_ca.clone()
    z_best = zij_trunk.clone()
    tm_best = 0.0
    best_step_idx = -1
    rollback_count = 0
    accepted_tm_history = []

    step_metrics = []
    t0 = time.time()
    stopped_early = False

    for step_idx in range(n_steps):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca,
            step_idx=step_idx,
        )

        # Composite score hesapla
        comp_score, comp_detail = composite_score_from_step(sr, weights)

        # Hard reject kontrolleri
        hard_reject = False
        reject_reason = ''
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            hard_reject = True
            reject_reason = f'Rg={sr.rg_ratio:.1f}>{CONFIDENCE_RG_MAX}'
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            hard_reject = True
            reject_reason = 'clash'
        if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
            hard_reject = True
            reject_reason = f'rmsd_init={sr.rmsd:.1f}>{CONFIDENCE_RMSD_INIT_MAX}'

        if hard_reject:
            accepted = False
        else:
            accepted = comp_score >= threshold
            if not accepted:
                reject_reason = f'comp={comp_score:.3f}<{threshold}'

        # Target metrikleri
        rmsd_tgt = compute_rmsd(sr.new_ca, target_ca)
        tm_tgt = tm_score(sr.new_ca, target_ca)

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'comp_score': comp_score,
            'rmsd_init': sr.rmsd,
            'rmsd_tgt': rmsd_tgt,
            'tm_tgt': tm_tgt,
            'ptm': sr.ptm,
            'plddt_mean': float(sr.plddt.mean()) if sr.plddt is not None else None,
            'ranking': sr.ranking_score,
            'mean_pae': sr.mean_pae,
            'rg_ratio': sr.rg_ratio,
            'contact_recon': sr.contact_recon,
            'contact_of3': sr.contact_of3,
            'has_clash': sr.has_clash,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
            'reject_reason': reject_reason,
            'rollback': False,
            # Selective mixing diagnostics
            'change_score_mean': sr.change_score_mean,
            'change_score_max': sr.change_score_max,
            'n_active_pairs': sr.n_active_pairs,
            'alpha_mask_mean': sr.alpha_mask_mean,
            **{f'd_{k}': v for k, v in comp_detail.items()},
        }

        if verbose:
            tag = 'ACCEPT' if accepted else 'REJECT'
            ptm_s = f'{sr.ptm:.3f}' if sr.ptm is not None else '-'
            pae_s = f'{sr.mean_pae:.1f}' if sr.mean_pae is not None else '-'
            rg_s = f'{sr.rg_ratio:.2f}' if sr.rg_ratio is not None else '-'
            sel_diag = ''
            if selective_mixing and sr.change_score_mean is not None:
                sel_diag = (
                    f'  cs={sr.change_score_mean:.3f}'
                    f' am={sr.alpha_mask_mean:.3f}'
                    f' np={sr.n_active_pairs}'
                )
            extra = f'  {reject_reason}' if not accepted else ''
            print(
                f'  [{step_idx+1:>2}/{n_steps}] {tag:<6}  '
                f'comp={comp_score:.3f}  '
                f'pTM={ptm_s}  mPAE={pae_s}  Rg={rg_s}  '
                f'RMSD_init={sr.rmsd:.1f}  '
                f'RMSD_tgt={rmsd_tgt:.2f}  TM={tm_tgt:.3f}  '
                f'FB={sr.fallback_level}  a={current_alpha:.3f}'
                f'{sel_diag}{extra}'
            )

        # Update state
        if accepted:
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha

            # Best-so-far rollback
            if tm_tgt > tm_best:
                tm_best = tm_tgt
                coords_best = sr.new_ca.clone()
                z_best = sr.z_modified.clone()
                best_step_idx = step_idx
            elif tm_best > 0 and tm_tgt < tm_best * (1.0 - BEST_ROLLBACK_TM_DROP):
                if verbose:
                    print(
                        f'  >> ROLLBACK: TM={tm_tgt:.3f} < best={tm_best:.3f}*'
                        f'{1.0-BEST_ROLLBACK_TM_DROP:.2f} -- '
                        f'step {best_step_idx+1}\'e geri donuluyor'
                    )
                coords_ca = coords_best.clone()
                z_current = z_best.clone()
                rollback_count += 1
                m['rollback'] = True

            # Adaptive early stopping
            accepted_tm_history.append(tm_tgt)
            if len(accepted_tm_history) >= ADAPTIVE_STOP_WINDOW + 1:
                recent = accepted_tm_history[-(ADAPTIVE_STOP_WINDOW + 1):]
                monoton_azalis = all(
                    recent[i] > recent[i + 1]
                    for i in range(len(recent) - 1)
                )
                if monoton_azalis:
                    if verbose:
                        tm_seq = ' -> '.join(f'{t:.3f}' for t in recent)
                        print(
                            f'  >> EARLY STOP: {ADAPTIVE_STOP_WINDOW} ardisik TM dusus: '
                            f'{tm_seq} -- best step {best_step_idx+1} donduruluyor'
                        )
                    stopped_early = True
                    m['early_stop'] = True
                    step_metrics.append(m)
                    break
        else:
            consecutive_rejected += 1
            if alpha_decay < 1.0:
                current_alpha = max(0.02, current_alpha * alpha_decay)
            if max_stall > 0 and consecutive_rejected >= max_stall:
                if verbose:
                    print(f'  STALL: {consecutive_rejected} consecutive rejected')
                break

        step_metrics.append(m)

        # Early abort
        if step_idx + 1 == early_abort_steps:
            n_acc = sum(1 for x in step_metrics if x['accepted'])
            if n_acc == 0:
                if verbose:
                    print(f'  EARLY ABORT: 0/{early_abort_steps} accepted in first steps')
                break

    wall = time.time() - t0
    total_steps = len(step_metrics)
    n_accepted = sum(1 for x in step_metrics if x['accepted'])

    # Best result
    accepted_metrics = [x for x in step_metrics if x['accepted']]
    if accepted_metrics:
        best_step = max(accepted_metrics, key=lambda x: x['tm_tgt'])
        best_tm = best_step['tm_tgt']
        best_rmsd = best_step['rmsd_tgt']
    else:
        best_tm = tm_score(initial_ca, target_ca)
        best_rmsd = compute_rmsd(initial_ca, target_ca)

    result = {
        'label': label,
        'threshold': threshold,
        'alpha': alpha,
        'alpha_decay': alpha_decay,
        'selective_mixing': selective_mixing,
        'selective_params': selective_params or {},
        'total_steps': total_steps,
        'accepted': n_accepted,
        'rejected': total_steps - n_accepted,
        'accept_rate': n_accepted / max(1, total_steps),
        'best_tm': best_tm,
        'best_rmsd': best_rmsd,
        'rollback_count': rollback_count,
        'stopped_early': stopped_early,
        'wall_s': wall,
        'step_metrics': step_metrics,
    }

    if verbose:
        print(f'\n  => {n_accepted}/{total_steps} accepted ({n_accepted/max(1,total_steps)*100:.0f}%)  '
              f'best_TM={best_tm:.4f}  best_RMSD={best_rmsd:.2f}  '
              f'rollbacks={rollback_count}  early_stop={stopped_early}  wall={wall:.0f}s')

    return result


# V4 best config (Phase A/B'den)
# NOT: Bu degerleri V4 sonuclarina gore guncelle!
V4_BEST_WEIGHTS = WEIGHT_PRESETS['A_ptm_heavy']
V4_BEST_THRESHOLD = 0.50
V4_BEST_ALPHA = 0.7
V4_BEST_DECAY = 0.8

ALL_RESULTS = {}
print('V5 Runner ready.')

## 5. Phase A: Uniform vs Selective (Baseline Karsilastirmasi)

V4'un en iyi config'i ile 2 run:
- A1: `selective_mixing=False` (uniform alpha, V4 baseline)
- A2: `selective_mixing=True` (default params: cutoff=0.1, base=0.0, max=1.0, linear)

Selective mixing'in baseline etkisini gormek icin hizli kontrol.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE A: UNIFORM vs SELECTIVE BASELINE
#  2 run — hizli A/B kontrol
# ════════════════════════════════════════════════════

ALL_RESULTS['phase_a'] = []

# A1: V4 baseline (uniform alpha)
r_a1 = run_v5(
    weights=V4_BEST_WEIGHTS,
    threshold=V4_BEST_THRESHOLD,
    label='A1_uniform_baseline',
    alpha=V4_BEST_ALPHA,
    alpha_decay=V4_BEST_DECAY,
    selective_mixing=False,
)
ALL_RESULTS['phase_a'].append(r_a1)

# A2: Selective mixing (default params)
r_a2 = run_v5(
    weights=V4_BEST_WEIGHTS,
    threshold=V4_BEST_THRESHOLD,
    label='A2_selective_default',
    alpha=V4_BEST_ALPHA,
    alpha_decay=V4_BEST_DECAY,
    selective_mixing=True,
    selective_params={
        'change_cutoff': 0.1,
        'alpha_base': 0.0,
        'alpha_max': 1.0,
        'mapping': 'linear',
        'w_connectivity': 0.5,
        'w_distance': 0.5,
    },
)
ALL_RESULTS['phase_a'].append(r_a2)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase_a']:
    rows.append({
        'label': r['label'],
        'selective': r['selective_mixing'],
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'rollbacks': r['rollback_count'],
        'early_stop': r['stopped_early'],
        'wall_s': f"{r['wall_s']:.0f}",
    })
df_a = pd.DataFrame(rows)
print('\n' + '='*80)
print('PHASE A: UNIFORM vs SELECTIVE BASELINE')
print('='*80)
print(df_a.to_string(index=False))

delta_tm = r_a2['best_tm'] - r_a1['best_tm']
print(f'\nSelective TM delta: {delta_tm:+.4f}')

## 6. Phase B: change_cutoff Sweep

Cutoff: hangi esik altindaki pair'ler alpha_base alacak?
Dusuk cutoff = daha fazla pair guncellenir, yuksek cutoff = daha secici.

Sabit: `alpha_base=0.0`, `alpha_max=1.0`, `mapping=linear`, `w=(0.5, 0.5)`

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE B: CHANGE_CUTOFF SWEEP
#  5 run: cutoff = [0.05, 0.1, 0.15, 0.2, 0.3]
# ════════════════════════════════════════════════════

ALL_RESULTS['phase_b'] = []

for cutoff in SELECTIVE_CHANGE_CUTOFFS:
    label = f'B_cutoff_{cutoff:.2f}'
    r = run_v5(
        weights=V4_BEST_WEIGHTS,
        threshold=V4_BEST_THRESHOLD,
        label=label,
        alpha=V4_BEST_ALPHA,
        alpha_decay=V4_BEST_DECAY,
        selective_mixing=True,
        selective_params={
            'change_cutoff': cutoff,
            'alpha_base': 0.0,
            'alpha_max': 1.0,
            'mapping': 'linear',
            'w_connectivity': 0.5,
            'w_distance': 0.5,
        },
    )
    ALL_RESULTS['phase_b'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase_b']:
    sp = r['selective_params']
    # Ortalama selective diagnostics (accepted step'lerden)
    acc_ms = [m for m in r['step_metrics'] if m['accepted'] and m.get('change_score_mean') is not None]
    avg_cs = np.mean([m['change_score_mean'] for m in acc_ms]) if acc_ms else 0
    avg_np = np.mean([m['n_active_pairs'] for m in acc_ms]) if acc_ms else 0
    avg_am = np.mean([m['alpha_mask_mean'] for m in acc_ms]) if acc_ms else 0
    rows.append({
        'label': r['label'],
        'cutoff': sp.get('change_cutoff', '-'),
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'avg_cs': f'{avg_cs:.3f}',
        'avg_np': f'{avg_np:.0f}',
        'avg_am': f'{avg_am:.3f}',
        'wall_s': f"{r['wall_s']:.0f}",
    })
df_b = pd.DataFrame(rows)
print('\n' + '='*90)
print('PHASE B: CHANGE_CUTOFF SWEEP')
print('='*90)
print(df_b.to_string(index=False))

best_b = max(ALL_RESULTS['phase_b'], key=lambda r: (r['best_tm'], r['accepted']))
BEST_CUTOFF = best_b['selective_params']['change_cutoff']
print(f'\nBest Phase B: {best_b["label"]} -- cutoff={BEST_CUTOFF} -- TM={best_b["best_tm"]:.4f}')

## 7. Phase C: alpha_base x alpha_max Sweep

Phase B'nin en iyi cutoff'u ile alpha aralik optimizasyonu.

- `alpha_base`: degismeyen pair'lerin alacagi alpha (`0.0` = tamamen koru, `0.05` = hafif guncelle)
- `alpha_max`: en cok degisen pair'lerin alacagi alpha (`0.5` - `1.0`)

Sabit: Phase B en iyi cutoff, `mapping=linear`, `w=(0.5, 0.5)`

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE C: ALPHA_BASE x ALPHA_MAX SWEEP
#  6 run: base=[0.0, 0.05] x max=[0.5, 0.7, 1.0]
# ════════════════════════════════════════════════════

ALL_RESULTS['phase_c'] = []

for alpha_base in SELECTIVE_ALPHA_BASES:
    for alpha_max in SELECTIVE_ALPHA_MAXES:
        label = f'C_base{alpha_base:.2f}_max{alpha_max:.1f}'
        r = run_v5(
            weights=V4_BEST_WEIGHTS,
            threshold=V4_BEST_THRESHOLD,
            label=label,
            alpha=V4_BEST_ALPHA,
            alpha_decay=V4_BEST_DECAY,
            selective_mixing=True,
            selective_params={
                'change_cutoff': BEST_CUTOFF,
                'alpha_base': alpha_base,
                'alpha_max': alpha_max,
                'mapping': 'linear',
                'w_connectivity': 0.5,
                'w_distance': 0.5,
            },
        )
        ALL_RESULTS['phase_c'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase_c']:
    sp = r['selective_params']
    acc_ms = [m for m in r['step_metrics'] if m['accepted'] and m.get('alpha_mask_mean') is not None]
    avg_am = np.mean([m['alpha_mask_mean'] for m in acc_ms]) if acc_ms else 0
    rows.append({
        'label': r['label'],
        'alpha_base': sp.get('alpha_base', '-'),
        'alpha_max': sp.get('alpha_max', '-'),
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'avg_am': f'{avg_am:.3f}',
        'rollbacks': r['rollback_count'],
        'wall_s': f"{r['wall_s']:.0f}",
    })
df_c = pd.DataFrame(rows)
print('\n' + '='*90)
print('PHASE C: ALPHA_BASE x ALPHA_MAX SWEEP')
print('='*90)
print(df_c.to_string(index=False))

best_c = max(ALL_RESULTS['phase_c'], key=lambda r: (r['best_tm'], r['accepted']))
BEST_ALPHA_BASE = best_c['selective_params']['alpha_base']
BEST_ALPHA_MAX = best_c['selective_params']['alpha_max']
print(f'\nBest Phase C: {best_c["label"]} -- '
      f'base={BEST_ALPHA_BASE} max={BEST_ALPHA_MAX} -- TM={best_c["best_tm"]:.4f}')

## 8. Phase D: Mapping & Weight Sweep

Phase B+C'nin en iyi parametreleri ile:
- **Mapping fonksiyonu**: `linear` vs `sigmoid` vs `step`
- **w_c / w_d agirliklari**: connectivity vs distance dengesi

3 mapping x 3 weight pair = 9 run

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE D: MAPPING & WEIGHT SWEEP
#  9 run: 3 mapping x 3 w_pair
# ════════════════════════════════════════════════════

ALL_RESULTS['phase_d'] = []

for mapping in SELECTIVE_MAPPINGS:
    for w_c, w_d in SELECTIVE_W_PAIRS:
        label = f'D_{mapping}_wc{w_c:.1f}_wd{w_d:.1f}'
        r = run_v5(
            weights=V4_BEST_WEIGHTS,
            threshold=V4_BEST_THRESHOLD,
            label=label,
            alpha=V4_BEST_ALPHA,
            alpha_decay=V4_BEST_DECAY,
            selective_mixing=True,
            selective_params={
                'change_cutoff': BEST_CUTOFF,
                'alpha_base': BEST_ALPHA_BASE,
                'alpha_max': BEST_ALPHA_MAX,
                'mapping': mapping,
                'w_connectivity': w_c,
                'w_distance': w_d,
            },
        )
        ALL_RESULTS['phase_d'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase_d']:
    sp = r['selective_params']
    acc_ms = [m for m in r['step_metrics'] if m['accepted'] and m.get('change_score_mean') is not None]
    avg_cs = np.mean([m['change_score_mean'] for m in acc_ms]) if acc_ms else 0
    avg_np = np.mean([m['n_active_pairs'] for m in acc_ms]) if acc_ms else 0
    rows.append({
        'label': r['label'],
        'mapping': sp.get('mapping', '-'),
        'w_c': sp.get('w_connectivity', '-'),
        'w_d': sp.get('w_distance', '-'),
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'avg_cs': f'{avg_cs:.3f}',
        'avg_np': f'{avg_np:.0f}',
        'rollbacks': r['rollback_count'],
        'wall_s': f"{r['wall_s']:.0f}",
    })
df_d = pd.DataFrame(rows)
print('\n' + '='*90)
print('PHASE D: MAPPING & WEIGHT SWEEP')
print('='*90)
print(df_d.to_string(index=False))

best_d = max(ALL_RESULTS['phase_d'], key=lambda r: (r['best_tm'], r['accepted']))
print(f'\nBest Phase D: {best_d["label"]} -- TM={best_d["best_tm"]:.4f}')

## 9. Analiz ve Grafikler

In [ ]:
# ════════════════════════════════════════════════════
#  ANALYSIS AND PLOTS - V5
# ════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120


# ── 1. TM-score karsilastirma tablosu (tum fazlar) ──
print('='*90)
print('GENEL SONUC TABLOSU')
print('='*90)
all_rows = []
for phase_name in ['phase_a', 'phase_b', 'phase_c', 'phase_d']:
    for r in ALL_RESULTS.get(phase_name, []):
        all_rows.append({
            'phase': phase_name,
            'label': r['label'],
            'selective': r['selective_mixing'],
            'best_TM': r['best_tm'],
            'best_RMSD': r['best_rmsd'],
            'accepted': r['accepted'],
            'accept%': f"{r['accept_rate']*100:.0f}%",
            'rollbacks': r['rollback_count'],
        })
df_all = pd.DataFrame(all_rows).sort_values('best_TM', ascending=False)
print(df_all.to_string(index=False))

# En iyi genel
all_results_flat = []
for phase_name in ['phase_a', 'phase_b', 'phase_c', 'phase_d']:
    all_results_flat.extend(ALL_RESULTS.get(phase_name, []))
best_overall = max(all_results_flat, key=lambda r: (r['best_tm'], r['accepted']))
print(f'\nEN IYI: {best_overall["label"]} -- TM={best_overall["best_tm"]:.4f} '
      f'RMSD={best_overall["best_rmsd"]:.2f}')


# ── 2. TM-score bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(all_results_flat)*0.3)))

labels = [r['label'][-35:] for r in all_results_flat]
tms = [r['best_tm'] for r in all_results_flat]
rmsds = [r['best_rmsd'] for r in all_results_flat]
colors = ['#e74c3c' if not r['selective_mixing'] else '#3498db' for r in all_results_flat]

axes[0].barh(labels, tms, color=colors)
axes[0].set_xlabel('Best TM-score')
axes[0].set_title('TM-score (kirmizi=uniform, mavi=selective)')
axes[0].axvline(tms[0], color='red', linestyle='--', alpha=0.5, label='V4 baseline')
axes[0].legend()

axes[1].barh(labels, rmsds, color=colors)
axes[1].set_xlabel('Best RMSD (A)')
axes[1].set_title('RMSD to target')

fig.suptitle('V5: Uniform vs Selective Mixing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# ── 3. Step trajectory (en iyi run) ──
def plot_step_trajectory_v5(result, title=''):
    metrics = result['step_metrics']
    steps = [m['step'] for m in metrics]
    comp = [m['comp_score'] for m in metrics]
    tm = [m['tm_tgt'] for m in metrics]
    accepted = [m['accepted'] for m in metrics]
    rollbacks = [m.get('rollback', False) for m in metrics]

    has_selective = any(m.get('change_score_mean') is not None for m in metrics)
    n_rows = 4 if has_selective else 2

    fig, axes = plt.subplots(n_rows, 1, figsize=(14, 4 * n_rows), sharex=True)

    # Composite score
    for s, c, a, rb in zip(steps, comp, accepted, rollbacks):
        marker = 'D' if rb else 'o'
        color = '#f39c12' if rb else ('#2ecc71' if a else '#e74c3c')
        axes[0].scatter(s, c, color=color, s=80, zorder=5, marker=marker,
                       edgecolors='black', linewidths=0.5)
    axes[0].plot(steps, comp, 'k-', alpha=0.3)
    axes[0].axhline(result.get('threshold', 0.50), color='orange', linestyle='--', label='threshold')
    axes[0].set_ylabel('Composite Score')
    axes[0].set_title(f'{title} -- Composite Score')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # TM-score
    for s, t, a, rb in zip(steps, tm, accepted, rollbacks):
        marker = 'D' if rb else 'o'
        color = '#f39c12' if rb else ('#2ecc71' if a else '#e74c3c')
        axes[1].scatter(s, t, color=color, s=80, zorder=5, marker=marker,
                       edgecolors='black', linewidths=0.5)
    axes[1].plot(steps, tm, 'k-', alpha=0.3)
    axes[1].set_ylabel('TM-score vs target')
    axes[1].set_title('TM-score vs Target')
    axes[1].grid(True, alpha=0.3)

    if has_selective:
        # change_score_mean trajectory
        cs_vals = [m.get('change_score_mean', 0) or 0 for m in metrics]
        axes[2].bar(steps, cs_vals, color='#8e44ad', alpha=0.7)
        axes[2].set_ylabel('change_score_mean')
        axes[2].set_title('Ortalama Degisim Skoru (pair basi)')
        axes[2].grid(True, alpha=0.3)

        # n_active_pairs trajectory
        np_vals = [m.get('n_active_pairs', 0) or 0 for m in metrics]
        axes[3].bar(steps, np_vals, color='#2980b9', alpha=0.7)
        axes[3].set_ylabel('n_active_pairs')
        axes[3].set_xlabel('Step')
        axes[3].set_title('Aktif Pair Sayisi (cutoff ustu)')
        axes[3].grid(True, alpha=0.3)
    else:
        axes[1].set_xlabel('Step')

    plt.tight_layout()
    plt.show()


# Baseline trajectory
if ALL_RESULTS.get('phase_a'):
    plot_step_trajectory_v5(ALL_RESULTS['phase_a'][0], 'A1: Uniform Baseline')
    if len(ALL_RESULTS['phase_a']) > 1:
        plot_step_trajectory_v5(ALL_RESULTS['phase_a'][1], 'A2: Selective Default')

# En iyi selective run
selective_results = [r for r in all_results_flat if r['selective_mixing']]
if selective_results:
    best_selective = max(selective_results, key=lambda r: r['best_tm'])
    plot_step_trajectory_v5(best_selective, f'Best Selective: {best_selective["label"]}')


# ── 4. Alpha mask heatmap (en iyi selective run, son accepted step) ──
# NOT: alpha_mask sadece pipeline icinde hesaplaniyor, burada StepResult diagnostics'ten
# summary istatistikler (mean, max, n_active) goruntuleniyor.
# Detayli heatmap icin pipeline'dan alpha_mask tensor'u save edilmeli.


# ── 5. V4 vs V5 best karsilastirmasi ──
print('\n' + '='*70)
print(f'{"V4 (uniform) vs V5 (selective) EN IYI KARSILASTIRMASI":^70}')
print('='*70)

v4_baseline = ALL_RESULTS['phase_a'][0]
v5_best = best_overall

print(f'{"Metrik":<25} {"V4 Uniform":>20} {"V5 Selective":>20}')
print(f'{"-"*70}')
print(f'{"Label":<25} {v4_baseline["label"]:>20} {v5_best["label"]:>20}')
print(f'{"Accept rate":<25} '
      f'{v4_baseline["accepted"]}/{v4_baseline["total_steps"]} '
      f'({v4_baseline["accept_rate"]*100:.0f}%):>20 '
      f'{v5_best["accepted"]}/{v5_best["total_steps"]} '
      f'({v5_best["accept_rate"]*100:.0f}%):>20')
print(f'{"Best TM-score":<25} {v4_baseline["best_tm"]:>20.4f} {v5_best["best_tm"]:>20.4f}')
print(f'{"Best RMSD (A)":<25} {v4_baseline["best_rmsd"]:>20.2f} {v5_best["best_rmsd"]:>20.2f}')
print(f'{"Rollbacks":<25} {v4_baseline["rollback_count"]:>20} {v5_best["rollback_count"]:>20}')
print(f'{"Early stop":<25} {str(v4_baseline["stopped_early"]):>20} {str(v5_best["stopped_early"]):>20}')
print(f'{"Wall time (s)":<25} {v4_baseline["wall_s"]:>20.0f} {v5_best["wall_s"]:>20.0f}')
print(f'{"-"*70}')
delta = v5_best['best_tm'] - v4_baseline['best_tm']
print(f'{"TM delta":<25} {"":>20} {delta:>+20.4f}')
print(f'{"="*70}')

if v5_best['selective_mixing']:
    sp = v5_best['selective_params']
    print(f'\nV5 en iyi selective params:')
    print(f'  change_cutoff  = {sp.get("change_cutoff", "-")}')
    print(f'  alpha_base     = {sp.get("alpha_base", "-")}')
    print(f'  alpha_max      = {sp.get("alpha_max", "-")}')
    print(f'  mapping        = {sp.get("mapping", "-")}')
    print(f'  w_connectivity = {sp.get("w_connectivity", "-")}')
    print(f'  w_distance     = {sp.get("w_distance", "-")}')

print('\nAnalysis complete.')

## 10. Sonuclari Drive'a Kaydet

In [ ]:
# ════════════════════════════════════════════════════
#  SAVE TO DRIVE
# ════════════════════════════════════════════════════
import json
from datetime import datetime

save_dir = '/content/drive/MyDrive/ANM-openfold3/search_v5'
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Serialize
save_data = {}
for phase_name, results in ALL_RESULTS.items():
    if isinstance(results, list):
        save_data[phase_name] = []
        for r in results:
            r_copy = {k: v for k, v in r.items()}
            save_data[phase_name].append(r_copy)
    else:
        save_data[phase_name] = results

# Save JSON
save_path = os.path.join(save_dir, f'results_{timestamp}.json')
with open(save_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print(f'JSON saved to {save_path}')

# Save summary CSV
csv_rows = []
for phase_name, results in ALL_RESULTS.items():
    if not isinstance(results, list):
        continue
    for r in results:
        sp = r.get('selective_params', {})
        csv_rows.append({
            'phase': phase_name,
            'label': r.get('label', ''),
            'selective_mixing': r.get('selective_mixing', False),
            'change_cutoff': sp.get('change_cutoff', ''),
            'alpha_base': sp.get('alpha_base', ''),
            'alpha_max': sp.get('alpha_max', ''),
            'mapping': sp.get('mapping', ''),
            'w_connectivity': sp.get('w_connectivity', ''),
            'w_distance': sp.get('w_distance', ''),
            'threshold': r.get('threshold', ''),
            'alpha': r.get('alpha', ''),
            'decay': r.get('alpha_decay', ''),
            'steps': r.get('total_steps', ''),
            'accepted': r.get('accepted', ''),
            'accept_rate': f"{r.get('accept_rate', 0)*100:.0f}%",
            'best_tm': f"{r.get('best_tm', 0):.4f}",
            'best_rmsd': f"{r.get('best_rmsd', 0):.2f}",
            'rollbacks': r.get('rollback_count', 0),
            'early_stop': r.get('stopped_early', False),
            'wall_s': f"{r.get('wall_s', 0):.0f}",
        })
csv_path = os.path.join(save_dir, f'summary_{timestamp}.csv')
pd.DataFrame(csv_rows).to_csv(csv_path, index=False)
print(f'CSV saved to {csv_path}')

print('\nDone! Sonuclari Drive\'dan indir ve analiz et.')